# **CODE GRAVEYARD**

In [ ]:
# scrna_data_viewer

#pca and batch unmixing

# STEP 3: Get PCs and batch unmix the data
#sc.pp.pca(live_redo)
sc.pp.pca(adata)
sc.pl.pca_variance_ratio(adata)
#pcs_l = live_redo.obsm['X_pca']
pcs_d = adata.obsm['X_pca']
#print(pcs.shape)  # (n_cells, n_pcs)

# Run Harmony on the PCA embedding
#harmony_out_l = hm.run_harmony(pcs_l, live_redo.obs, "sample",max_iter_harmony=25)
harmony_out_d = hm.run_harmony(pcs_d, adata.obs,"sample",max_iter_harmony=25)


print("finished harmony")

# Store corrected PCs back in the AnnData object
#live_redo.obsm['X_pca_harmony'] = harmony_out_l.Z_corr
adata.obsm['X_pca_harmony'] = harmony_out_d.Z_corr
#sc.pp.neighbors(live_redo, use_rep='X_pca_harmony')
sc.pp.neighbors(adata, use_rep='X_pca_harmony')
#sc.tl.umap(live_redo)
sc.tl.umap(adata)

In [ ]:
#scRNA_part1_cellranger

#soupx

adatasr = {}

for sample_id, filename in raw_samples.items():
    sample_adata = sc.read_10x_h5(filename)
    #sample_adata = sc.read_10x_h5(filename)
    sample_adata.var_names_make_unique()
    adatasr[sample_id] = sample_adata

adatar = ad.concat(adatas, label="sample")
adatar.obs['treatment'] = '3DPI'
adatar.obs_names_make_unique()

adatar.obs['treatment'] = "3DPI"
for val in range(1,17):
    match val:
            case val if 1<=int(val)<=5:
                adatar.obs.loc[adatar.obs["sample"] == f"sample_{val}", 'treatment'] = 'no_fiber'
            case val if 6<=int(val)<=10:
                adatar.obs.loc[adatar.obs["sample"] == f"sample_{val}", 'treatment'] = 'amp'
            case _:
                adatar.obs.loc[adatar.obs["sample"] == f"sample_{val}", 'treatment'] = '3DPI'
#Display our AnnData object via observations
adatar.shape

In [ ]:
#scRNA_part2_analysis

#an attempt at pathway analysis

all_files = []

my_prefix = "/scratch/alpine/hvose@xsede.org/viralogue-pipelines/scRNA-seq-pipeline/20260313_cluster_comparisons/"
cluster_file = "all_cluster_comparisons.txt"
with open(cluster_file, "r", encoding="utf-8") as f:
    for line in f.readlines():
        print(f"appending line: {my_prefix}{line}")
        all_files.append(f"{my_prefix}{line}".strip())
f.close()

for item in all_files:
    up_down_dict = {"up":[],"down":[]}
    with open(item, "r", encoding="utf-8") as curr_file:
        for idx, curr_line in enumerate(curr_file.readlines()):
            my_line = curr_line.strip().split(",")
            #print(my_line)
            if idx != 0:
                e_check = re.search(f"(.+)e(.+)",my_line[4])
                if e_check is not None:
                    my_line[4] = float(e_check.group(1)) * (10**float(e_check.group(2)))
                    print(f"new e_check is {my_line[4]}")
                    
                if (float(my_line[4]) < 0.05) and (float(my_line[2]) < 0):
                    up_down_dict["down"].append(my_line[0])
                elif (float(my_line[4]) < 0.05) and (float(my_line[2]) > 0):
                    up_down_dict["up"].append(my_line[0])

        if len(up_down_dict["up"]) > 0:
            up_query = sc.queries.enrich(up_down_dict["up"])
            up_query.to_csv(f"{item}_GO_up.csv")
        else:
            print(f"WARNING! No significant pathways found for the up regulation in {item}")
            up_query = ["No Valid Pathways Found"]

        if len(up_down_dict["down"]) > 0:
            down_query = sc.queries.enrich(up_down_dict["down"])
            down_query.to_csv(f"{item}_GO_down.csv")
        else:
            print(f"WARNING! No significant pathways found for the down regulation in {item}")
            down_query = ["No Valid Pathways Found"]

print("uncomment when needed")


        # for val in up_down_dict['up']:
        #     print(val)
        # print(up_down_dict["down"])
        
                    

#my_query = sc.queries.enrich(["Muc2","Spink4","Tff3","Agr2"],org="mmusculus")
#type(my_query)

In [ ]:
#scRNA_part2_analysis

# PATHWAY ANALYSIS LOOP
df_dict = {}

curr_dataset = "condition_exp"
curr_cat = "leiden_res_0.50"
remove_overlap = True

stop = max(int(item) for item in adata_filtered.obs[curr_cat].cat.categories.tolist()) + 1

for i in range(0,stop):
    curr_concats = []
    test_subset_mhvy = adata_filtered[(adata_filtered.obs[curr_cat] == str(i)) & (adata_filtered.obs['condition'].isin(['Mock','MHVY']))].copy()
    test_subset_dhe = adata_filtered[(adata_filtered.obs[curr_cat] == str(i)) & (adata_filtered.obs['condition'].isin(['Mock','dHE']))].copy()
    
    test_subset_mhvy.obs['treat_cat'] = pd.Categorical(test_subset_mhvy.obs['condition'], categories=["Mock", "MHVY"], ordered=True)
    test_subset_dhe.obs['treat_cat'] = pd.Categorical(test_subset_dhe.obs['condition'], categories=["Mock", "dHE"], ordered=True)
    indices_mhvy = test_subset_mhvy.obs.sort_values(['treat_cat']).index
    indices_dhe = test_subset_dhe.obs.sort_values(['treat_cat']).index
    mhvy = test_subset_mhvy[indices_mhvy,:]
    dhe = test_subset_dhe[indices_dhe,:]
    #generate ranked genes for this specific group

    if mhvy[mhvy.obs["condition"] == "Mock"].n_obs > 1:
        sc.tl.rank_genes_groups(mhvy,
                                groupby='treat_cat',
                                use_raw=False,
                                method='wilcoxon',
                                groups=['MHVY'],
                                reference='Mock')
        
    if dhe[dhe.obs["condition"] == "Mock"].n_obs > 1:
        sc.tl.rank_genes_groups(dhe,
                                groupby='treat_cat',
                                use_raw=False,
                                method='wilcoxon',
                                groups=['dHE'],
                                reference='Mock')

         # get deg result
        result_mhvy = mhvy.uns['rank_genes_groups']
        result_dhe = dhe.uns['rank_genes_groups']
        mhvy_groups = result_mhvy['names'].dtype.names
        dhe_groups = result_dhe['names'].dtype.names
        mhvy_degs = pd.DataFrame(
            {key: result_mhvy[key][group] for group in mhvy_groups for key in ['names','scores', 'pvals','pvals_adj','logfoldchanges']})
        dhe_degs = pd.DataFrame(
            {key: result_dhe[key][group] for group in dhe_groups for key in ['names','scores', 'pvals','pvals_adj','logfoldchanges']})
        mhvy_degs_sig = mhvy_degs[mhvy_degs.pvals_adj < 0.05]
        dhe_degs_sig = dhe_degs[dhe_degs.pvals_adj <= 0.05]
        mhvy_degs_up = mhvy_degs_sig[mhvy_degs_sig.logfoldchanges > 0]
        mhvy_degs_dw = mhvy_degs_sig[mhvy_degs_sig.logfoldchanges < 0]
        dhe_degs_up = dhe_degs_sig[dhe_degs_sig.logfoldchanges > 0]
        dhe_degs_dw = dhe_degs_sig[dhe_degs_sig.logfoldchanges < 0]
        up_both = []
        dw_both = []

        if remove_overlap:
            for m_gene in mhvy_degs_up:
                if m_gene in dhe_degs_up:
                    del mhvy_degs_up[m_gene]
                    del dhe_degs_up[m_gene]
                    up_both.append(m_gene)

            for dm_gene in mhvy_degs_dw:
                if m_gene in dhe_degs_dw:
                    del mhvy_degs_dw[dm_gene]
                    del dhe_degs_dw[dm_gene]
                    dw_both.append(dm_gene)


    
#         if len(mhvy_degs_up) > 1:
#             m_enr_up_process = gp.enrichr(mhvy_degs_up.names,
#                                 gene_sets='GO_Biological_Process_2025',
#                                 outdir=None)
#             m_enr_up_molecular = gp.enrichr(mhvy_degs_up.names,
#                             gene_sets='GO_Molecular_Function_2025',
#                             outdir=None)

#             d_enr_up_process = gp.enrichr(dhe_degs_up.names,
#                                 gene_sets='GO_Biological_Process_2025',
#                                 outdir=None)
#             d_enr_up_molecular = gp.enrichr(dhe_degs_up.names,
#                             gene_sets='GO_Molecular_Function_2025',
#                             outdir=None)
            
#             m_enr_up_process.res2d['UP_DW'] = f"UP_MHVY_PROCESS_{i}"
#             m_enr_up_molecular.res2d['UP_DW'] = f"UP_MHVY_MOLEC_{i}"
#             d_enr_up_process.res2d['UP_DW'] = f"UP_dHE_PROCESS_{i}"
#             d_enr_up_molecular.res2d['UP_DW'] = f"UP_dHE_MOLEC_{i}"
#             curr_concats.append(m_enr_up_process.res2d)
#             curr_concats.append(m_enr_up_molecular.res2d)
#             curr_concats.append(d_enr_up_process.res2d)
#             curr_concats.append(d_enr_up_molecular.res2d)
#         else:
#             print(f"WARNING: No upregulated DEG for cluster {i} in condition {condition} because the gene name contains 1 or fewer genes.")
            
    
#         if len(mhvy_degs_dw) > 1:
#             m_enr_dw_process = gp.enrichr(mhvy_degs_dw.names,
#                                 gene_sets='GO_Biological_Process_2025',
#                                 outdir=None)
#             m_enr_dw_molecular = gp.enrichr(mhvy_degs_dw.names,
#                             gene_sets='GO_Molecular_Function_2025',
#                             outdir=None)
#             d_enr_dw_process = gp.enrichr(dhe_degs_dw.names,
#                                 gene_sets='GO_Biological_Process_2025',
#                                 outdir=None)
#             d_enr_dw_molecular = gp.enrichr(dhe_degs_dw.names,
#                             gene_sets='GO_Molecular_Function_2025',
#                             outdir=None)
#             m_enr_dw_process.res2d['UP_DW'] = f"DOWN_MHVY_PROCESS_{i}"
#             m_enr_dw_molecular.res2d['UP_DW'] = f"DOWN_MHVY_MOLEC_{i}"
#             d_enr_dw_process.res2d['UP_DW'] = f"DOWN_dHE_PROCESS_{i}"
#             d_enr_dw_molecular.res2d['UP_DW'] = f"DOWN_dHE_MOLEC_{i}"
#             curr_concats.append(m_enr_dw_process.res2d)
#             curr_concats.append(m_enr_dw_molecular.res2d)
#             curr_concats.append(d_enr_dw_process.res2d)
#             curr_concats.append(d_enr_dw_molecular.res2d)
#         else:
#             print(f"WARNING: No downregulated DEG for cluster {i} in condition {condition} because the gene name contains 1 or fewer genes.")
        
#         # enr_up_cellular = gp.enrichr(degs_up.names,
#         #                     gene_sets='GO_Cellular_Component_2025',
#         #                     outdir=None)
#         # enr_dw_cellular = gp.enrichr(degs_dw.names,
#         #                     gene_sets='GO_Cellular_Component_2025',
#         #                     outdir=None)
        
        
#         # concat results
#         if len(curr_concats) > 0:
#             enr_res = pd.concat(curr_concats)
#             df_dict[f"{i}_{curr_condition}"] = enr_res
#         #ax = gp.barplot(enr_res, figsize=(3,5),
#         #                group ='UP_DW',
#         #                title ="GO_BP",
#         #                color = ['b','r'])
    
#         print(f"{i} list for metascape:")

#         print(mhvy_degs_up)
#         print(mhvy_degs_dw)
#         print(dhe_degs_up)
#         print(dhe_degs_dw)
#         print(up_both)
#         print(dw_both)
                
#     else:
#         print(f"WARNING! No cells found for cluster {i} in condition {condition}")
    
   
# final_enr_res = None

# for values in df_dict.values():
#     if final_enr_res is None:
#         final_enr_res = values
#     else:
#         final_enr_res = pd.concat([final_enr_res,values])

# final_enr_res.to_csv(f"{curr_dataset}_fullcompare_vs_Mock.csv")


In [2]:
#scRNA_part2_analysis

#misc: exon analysis

reg_diff_exp = "/scratch/alpine/hvose@xsede.org/bulkDiffExpression_04112026.txt"
exon_diff_exp = "/scratch/alpine/hvose@xsede.org/DiffExpressionLocus_Mock_MHVY_dHE_Exon_count_03252026.txt"

cols = ['gene','mock_vs_mhvy','mock_vs_dhe']

diff_exp_df = pd.DataFrame(columns=cols)

with open(reg_diff_exp,"r",encoding="utf-8") as f:
    for idx,line in enumerate(f.readlines()):
        if idx != 0:
            my_line = line.strip().split()
            new_row = pd.DataFrame({'gene': [my_line[0]],'mock_vs_mhvy': [float(my_line[24]) if my_line[24] != "NA" else 0], 'mock_vs_dhe': [float(my_line[27]) if my_line[27] != "NA" else 0]})
            diff_exp_df = pd.concat([new_row, diff_exp_df])
f.close()

filtered_exp_df = diff_exp_df[(diff_exp_df['mock_vs_mhvy'] < 0.05) | (diff_exp_df['mock_vs_dhe'] < 0.05)]
print(filtered_exp_df.head())

diff_exon_list = []

with open(exon_diff_exp,"r",encoding="utf-8") as e:
    for idx, line in enumerate(e.readlines()):
        if idx != 0:
            my_line = line.strip().split()
            if (my_line[24] != "NA") | (my_line[27] != "NA"):
                if (float(my_line[24]) < 0.05) | (float(my_line[27]) < 0.05):
                    exon = re.search(r"(.+)--",my_line[0]).group(1)
                    if exon not in filtered_exp_df['gene']:
                        diff_exon_list.append(my_line[0])
e.close()

diff_exon_list.sort()

diff_hold = diff_exon_list


In [ ]:
#scRNA_part2_analysis
#diff_exon_list = ['NM_001025311--Part3', 'NM_001025311--Part4', 'NM_001025311--Part6', 'NM_001025311--Part7']
diff_exon_list = diff_hold
curr_exon = "A--Part1"
curr_part = 1
actual_part = 1
maybe_full = []
exon_list = []
partial = []
threshold = 3
for idx, exon in enumerate(diff_exon_list):
    #print(f"checking {exon} against curr_exon {curr_exon}")
    gene_name = re.search(r"(.+)--Part(\d+)",exon)
    prefix = re.search(r"(.+)--Part(\d+)",curr_exon)
    if (prefix.group(1) != gene_name.group(1)) and (idx != 0):
        #print(f"DIFFERENT! {prefix.group(1)} != {gene_name.group(1)}")
        curr_exon = exon
        if curr_part >= threshold:
            print(f"curr_part for prefix {prefix.group(1)} is {curr_part}")
            maybe_full.append(exon_list)
        else:
            partial.append(exon_list)
        exon_list = [exon]
        curr_part = 1
        actual_part = int(gene_name.group(2))
    elif idx == 0:
        curr_exon = exon
        exon_list.append(curr_exon)
    else:
        curr_exon = exon
        curr_part += 1
        actual_part = int(gene_name.group(2))
        exon_list.append(curr_exon)
        if curr_part != actual_part:
            curr_part = 1

if curr_part >= threshold:
    maybe_full.append(exon_list)
else:
    partial.append(exon_list)

#print(diff_exon_list)

maybe_full.sort()
partial.sort()

print("MAYBE FULL")
for mf in maybe_full:
    print(mf)

print("PARTIAL")
for par in partial:
    par_match = re.search(r"(.+)--Part(\d+)",par[0])
    print(par_match.group(1))
            

In [ ]:
#scRNA_part2_analysis

#individual cluster comparison DEGs

adata_goblet = adata_filtered[(adata_filtered.obs["cluster_cell_type"] == "Goblet Cell")]

sc.tl.rank_genes_groups(adata_goblet, groupby='group', reference='5DPI', method='wilcoxon',key_added='i_type_compare_4')
sc.get.rank_genes_groups_df(adata_goblet, group="Mock",key='i_type_compare_4')
sc.pl.rank_genes_groups_dotplot(
    adata_goblet, groupby="group", standard_scale="var", n_genes=10, key="i_type_compare_4", cmap=dp_cmap,save="Mock_goblet.svg"
)